# Indian city at coarse resolution: Sentinel-2 (10 m) over Hyderabad, zero-shot + fine-tuning with OpenStreetMap labels
Backup for LISS-IV (no login needed). Inputs: `rr-code`, `segformer-ft-model`. Settings: GPU T4 x2, **Internet On**. Save Version → Save & Run All.

In [1]:
CENTER = "17.385,78.487"      # Hyderabad.   Vijayawada: "16.506,80.648"
SIZE_KM = 24

In [2]:
import glob, os, shutil
hits = glob.glob('/kaggle/input/**/scripts/evaluate_checkpoint.py', recursive=True)
assert hits, "rr-code dataset is not attached"
PROJECT = os.path.dirname(os.path.dirname(hits[0]))
shutil.copytree(PROJECT, '/kaggle/working/rr', dirs_exist_ok=True)
os.chdir('/kaggle/working/rr')
print("PROJECT:", PROJECT)
!pip -q install segmentation-models-pytorch osmnx rasterio geopandas pystac-client
!nvidia-smi -L

PROJECT: /kaggle/input/datasets/kashaksingh/rr-code/route-resilience-ai
GPU 0: Tesla T4 (UUID: GPU-2f8f03f1-99b9-055f-e141-dd2198fb4b94)
GPU 1: Tesla T4 (UUID: GPU-1f371b3d-6592-596b-2fb7-115122696e96)


In [3]:
%%writefile /kaggle/working/rr/scripts/fetch_sentinel2.py
"""
Download a cloud-free Sentinel-2 L2A RGB window around a point, no login needed.

    python scripts/fetch_sentinel2.py --center 17.385,78.487 --size-km 24 --out data/s2/hyderabad.tif

Uses the public Earth Search STAC catalogue (Element 84, AWS open data) and
reads only the requested window from the Cloud-Optimised GeoTIFFs, so a 24 km
window costs a few MB instead of a full 110 km tile.
Output: one 3-band GeoTIFF (R, G, B = B04, B03, B02) at 10 m in the tile's UTM CRS.
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path

import numpy as np

STAC = "https://earth-search.aws.element84.com/v1"


def main() -> None:
    import rasterio
    from pystac_client import Client
    from rasterio.warp import transform_bounds
    from rasterio.windows import from_bounds

    ap = argparse.ArgumentParser()
    ap.add_argument("--center", default="17.385,78.487", help="lat,lon (default Hyderabad)")
    ap.add_argument("--size-km", type=float, default=24.0)
    ap.add_argument("--start", default="2025-11-01")
    ap.add_argument("--end", default="2026-04-30")
    ap.add_argument("--max-cloud", type=float, default=10.0)
    ap.add_argument("--out", default="data/s2/scene.tif")
    a = ap.parse_args()

    lat, lon = (float(v) for v in a.center.split(","))
    dlat = a.size_km / 2 / 111.32
    dlon = dlat / np.cos(np.radians(lat))
    bbox = (lon - dlon, lat - dlat, lon + dlon, lat + dlat)

    items = list(Client.open(STAC).search(
        collections=["sentinel-2-l2a"], bbox=bbox, datetime=f"{a.start}/{a.end}",
        query={"eo:cloud_cover": {"lt": a.max_cloud}}, max_items=60).items())
    if not items:
        raise SystemExit("No Sentinel-2 scene below the cloud limit - raise --max-cloud.")

    def coverage(it):
        from shapely.geometry import box, shape
        g = shape(it.geometry)
        return g.intersection(box(*bbox)).area / box(*bbox).area

    items.sort(key=lambda it: (-round(coverage(it), 2), it.properties["eo:cloud_cover"]))
    item = items[0]
    print(f"[s2] {item.id}  cloud {item.properties['eo:cloud_cover']:.1f}%  "
          f"coverage {coverage(item):.0%}  date {item.datetime:%Y-%m-%d}")

    bands, meta = [], None
    for key in ("red", "green", "blue"):
        href = item.assets[key].href
        with rasterio.open(href) as src:
            b = transform_bounds("EPSG:4326", src.crs, *bbox)
            win = from_bounds(*b, transform=src.transform).round_offsets().round_lengths()
            bands.append(src.read(1, window=win))
            if meta is None:
                meta = {"crs": src.crs, "transform": src.window_transform(win)}
    arr = np.stack(bands)
    out = Path(a.out)
    out.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(out, "w", driver="GTiff", height=arr.shape[1], width=arr.shape[2],
                       count=3, dtype=arr.dtype, crs=meta["crs"],
                       transform=meta["transform"], compress="deflate") as d:
        d.write(arr)
    (out.with_suffix(".json")).write_text(json.dumps({
        "item": item.id, "date": str(item.datetime), "cloud_cover": item.properties["eo:cloud_cover"],
        "bbox_wgs84": bbox, "bands": ["B04 red", "B03 green", "B02 blue"],
        "source": "Copernicus Sentinel-2 L2A via Earth Search (AWS open data)"}, indent=2))
    print(f"[s2] wrote {out}  {arr.shape[2]}x{arr.shape[1]} px")


if __name__ == "__main__":
    main()


Writing /kaggle/working/rr/scripts/fetch_sentinel2.py


In [4]:
%%writefile /kaggle/working/rr/scripts/prepare_liss4.py
"""
Build a LISS-IV (or Sentinel-2) road dataset with OpenStreetMap labels.

    python scripts/prepare_liss4.py --scene <folder or .tif> --out data/processed_liss4 \
        --center 17.385,78.487 --size-km 24

ResourceSat-2/2A LISS-IV (5.8 m, bands G/R/NIR, no blue) as delivered by
Bhoonidhi: either one multi-band GeoTIFF or one GeoTIFF per band (BAND2/3/4).

Steps
  1. find and stack the three bands, cut a square window around --center
     (falls back to the scene centre if the point is outside the scene);
  2. upsample by --upsample (default 2 -> 2.9 m/px) so 1-2 px roads become
     learnable by a model with a stride-4 encoder;
  3. fetch OSM roads for the window and rasterise them on the same grid
     (src/data/osm_groundtruth.py); roads narrower than --min-width-m are
     dropped as unresolvable at this resolution;
  4. cut tiles and split them SPATIALLY (west -> train, then val, east -> test)
     so test tiles never neighbour a training tile;
  5. write the same layout as DeepGlobe preparation, so kaggle_train.py and
     evaluate_checkpoint.py run on it unchanged.
"""
from __future__ import annotations

import argparse
import json
import re
import sys
from pathlib import Path

import numpy as np

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))


def find_bands(scene: Path) -> list[Path]:
    """Return [G, R, NIR] GeoTIFF paths, or [single multi-band file]."""
    tifs = sorted([p for p in (scene.rglob("*") if scene.is_dir() else [scene])
                   if p.suffix.lower() in (".tif", ".tiff")])
    if not tifs:
        raise SystemExit(f"No GeoTIFF found under {scene}")
    by_band = {}
    for p in tifs:
        m = re.search(r"(?:BAND|_B)(\d)", p.stem.upper())
        if m:
            by_band[int(m.group(1))] = p
    if all(b in by_band for b in (2, 3, 4)):
        return [by_band[2], by_band[3], by_band[4]]           # G, R, NIR
    import rasterio

    for p in sorted(tifs, key=lambda q: -q.stat().st_size):
        with rasterio.open(p) as s:
            if s.count >= 3:
                return [p]
    raise SystemExit("Could not identify LISS-IV bands. Files found:\n  "
                     + "\n  ".join(str(t) for t in tifs))


def read_window(bands: list[Path], center: tuple | None, size_km: float, upsample: float):
    import rasterio
    from rasterio.enums import Resampling
    from rasterio.warp import transform as warp_xy
    from rasterio.windows import Window

    with rasterio.open(bands[0]) as src:
        crs, tr, W, H = src.crs, src.transform, src.width, src.height
        gsd = abs(tr.a)
        if crs.is_geographic:
            raise SystemExit("Scene is in lat/lon; expected a projected (UTM) ortho product.")
        if center is not None:
            xs, ys = warp_xy("EPSG:4326", crs, [center[1]], [center[0]])
            col, row = ~tr * (xs[0], ys[0])
        else:
            col, row = W / 2, H / 2
        inside = 0 <= col < W and 0 <= row < H
        if not inside:
            print(f"[liss4] centre {center} is outside this scene - using the scene centre")
            col, row = W / 2, H / 2
        half = int(size_km * 1000 / gsd / 2)
        c0, r0 = int(max(col - half, 0)), int(max(row - half, 0))
        c1, r1 = int(min(col + half, W)), int(min(row + half, H))
        win = Window(c0, r0, c1 - c0, r1 - r0)
        out_shape = (int(win.height * upsample), int(win.width * upsample))
        wtr = rasterio.windows.transform(win, tr) * rasterio.Affine.scale(1 / upsample)

    arrs = []
    if len(bands) == 1:
        with rasterio.open(bands[0]) as src:
            a = src.read([1, 2, 3], window=win, out_shape=(3, *out_shape),
                         resampling=Resampling.bilinear)
            arrs = list(a)
    else:
        for b in bands:
            with rasterio.open(b) as src:
                arrs.append(src.read(1, window=win, out_shape=out_shape,
                                     resampling=Resampling.bilinear))
    stack = np.stack(arrs, -1).astype(np.float32)               # G, R, NIR
    return {"array": stack, "transform": wtr, "crs": crs, "gsd": gsd / upsample,
            "native_gsd": gsd, "window": [int(c0), int(r0), int(c1), int(r1)]}


def to_display_rgb(stack: np.ndarray, sensor: str = "liss4") -> tuple[np.ndarray, np.ndarray]:
    """Per-band 1-99 percentile stretch (ignoring nodata zeros). LISS-IV bands
    (G, R, NIR) go through the project's pseudo-RGB mapping (no blue band);
    Sentinel-2 input is already R, G, B."""
    from src.data.geo_io import select_rgb

    valid = stack.sum(-1) > 0
    out = np.zeros_like(stack)
    for i in range(3):
        v = stack[..., i][valid]
        lo, hi = (np.percentile(v, [1, 99]) if v.size else (0, 1))
        out[..., i] = np.clip((stack[..., i] - lo) / max(hi - lo, 1e-6), 0, 1)
    return (select_rgb(out, "liss4") if sensor == "liss4" else out), valid


def osm_mask(win: dict, cache: Path, min_width_m: float, sensor: str = "liss4") -> dict:
    import rasterio
    from rasterio.warp import transform_bounds

    from src.data.osm_groundtruth import download_osm_roads, rasterize_roads

    h, w = win["array"].shape[:2]
    tmp = cache / "window_grid.tif"
    cache.mkdir(parents=True, exist_ok=True)
    with rasterio.open(tmp, "w", driver="GTiff", height=h, width=w, count=1,
                       dtype="uint8", crs=win["crs"], transform=win["transform"]) as d:
        d.write(np.zeros((1, h, w), np.uint8))
        b = transform_bounds(win["crs"], "EPSG:4326", *d.bounds, densify_pts=21)
    edges = download_osm_roads(b, network_type="drive", cache_dir=cache)
    rep = rasterize_roads(edges, tmp, sensor=sensor, min_resolvable_width_m=min_width_m)
    rep["bounds_wgs84"] = list(b)
    return rep


def main() -> None:
    import cv2

    ap = argparse.ArgumentParser()
    ap.add_argument("--scene", required=True, help="folder with the LISS-IV product, or a .tif")
    ap.add_argument("--sensor", default="liss4", choices=["liss4", "sentinel2"],
                    help="liss4: bands G,R,NIR (5.8 m). sentinel2: one RGB GeoTIFF (10 m) "
                         "from scripts/fetch_sentinel2.py")
    ap.add_argument("--out", default="data/processed_liss4")
    ap.add_argument("--center", default="17.385,78.487",
                    help="lat,lon of the study area (default: Hyderabad)")
    ap.add_argument("--size-km", type=float, default=24.0)
    ap.add_argument("--upsample", type=float, default=2.0)
    ap.add_argument("--tile", type=int, default=512)
    ap.add_argument("--min-width-m", type=float, default=6.0,
                    help="drop OSM roads narrower than this (unresolvable at 5.8 m)")
    ap.add_argument("--val-frac", type=float, default=0.15)
    ap.add_argument("--test-frac", type=float, default=0.2)
    ap.add_argument("--cache", default="data/osm")
    args = ap.parse_args()

    center = tuple(float(v) for v in args.center.split(",")) if args.center else None
    bands = find_bands(Path(args.scene))
    print("[liss4] bands:", [b.name for b in bands])
    win = read_window(bands, center, args.size_km, args.upsample)
    rgb, valid = to_display_rgb(win["array"], args.sensor)
    print(f"[liss4] window {rgb.shape[1]}x{rgb.shape[0]} px at {win['gsd']:.2f} m/px "
          f"(native {win['native_gsd']:.2f} m)")

    lab = osm_mask(win, Path(args.cache), args.min_width_m, args.sensor)
    mask = lab["mask"] & valid
    print(f"[liss4] OSM roads kept={lab['kept']} dropped={lab['dropped']} "
          f"road pixels={mask.mean():.2%}")
    if mask.mean() < 0.003:
        raise SystemExit("Almost no OSM roads in this window - check --center / CRS.")

    out = Path(args.out)
    T = args.tile
    H, W = mask.shape
    cols = W // T
    n_test = max(1, int(round(cols * args.test_frac)))
    n_val = max(1, int(round(cols * args.val_frac)))
    counts = {"train": 0, "val": 0, "test": 0}
    for r in range(0, H - T + 1, T):
        for ci, c in enumerate(range(0, W - T + 1, T)):
            v = valid[r:r + T, c:c + T]
            if v.mean() < 0.9:
                continue
            split = ("test" if ci >= cols - n_test else
                     "val" if ci >= cols - n_test - n_val else "train")
            stem = f"{args.sensor}_{r // T:02d}_{ci:02d}"
            di, dm = out / split / "images", out / split / "masks"
            di.mkdir(parents=True, exist_ok=True)
            dm.mkdir(parents=True, exist_ok=True)
            im = (rgb[r:r + T, c:c + T] * 255).astype(np.uint8)
            cv2.imwrite(str(di / f"{stem}.png"), im[..., ::-1])
            cv2.imwrite(str(dm / f"{stem}.png"), mask[r:r + T, c:c + T].astype(np.uint8) * 255)
            counts[split] += 1
    print("[liss4] tiles:", counts)

    prev = (rgb * 255).astype(np.uint8).copy()
    prev[mask] = (0.4 * prev[mask] + 0.6 * np.array([255, 210, 0])).astype(np.uint8)
    s = 1600 / max(prev.shape[:2])
    cv2.imwrite(str(out / "preview_osm_overlay.jpg"),
                cv2.resize(prev, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)[..., ::-1])

    (out / "manifest.json").write_text(json.dumps({
        "source": ("ResourceSat-2/2A LISS-IV (Bhoonidhi, NRSC/ISRO)" if args.sensor == "liss4"
                   else "Copernicus Sentinel-2 L2A (10 m RGB)"),
        "sensor": args.sensor,
        "bands": [str(b) for b in bands], "center_latlon": center,
        "size_km": args.size_km, "upsample": args.upsample,
        "gsd_m": win["gsd"], "native_gsd_m": win["native_gsd"],
        "labels": "OpenStreetMap drive network, buffered by road class",
        "min_width_m": args.min_width_m, "osm": {k: v for k, v in lab.items() if k != "mask"},
        "split": "spatial columns: west=train, then val, east=test", "counts": counts,
    }, indent=2, default=str))
    print(f"[liss4] wrote {out}")


if __name__ == "__main__":
    main()


Writing /kaggle/working/rr/scripts/prepare_liss4.py


In [5]:
import torch
chosen = None
for c in glob.glob('/kaggle/input/**/best.pt', recursive=True):
    ck = torch.load(c, map_location="cpu", weights_only=False)
    if ck.get("args", {}).get("model") == "segformer_b3" and ck.get("epoch") == 21:
        chosen = c
assert chosen, "Fine-tuned SegFormer (epoch 21) not found - attach segformer-ft-model"
print("MODEL:", chosen)

MODEL: /kaggle/input/datasets/kashaksingh/rr-code/route-resilience-ai/checkpoints/segformer_b3_ft/best.pt


In [6]:
!python scripts/fetch_sentinel2.py --center "$CENTER" --size-km $SIZE_KM --out /kaggle/working/s2/scene.tif

[s2] S2A_43QHV_20260429_0_L2A  cloud 0.0%  coverage 100%  date 2026-04-29
[s2] wrote /kaggle/working/s2/scene.tif  2447x2433 px


In [7]:
P = "/kaggle/working/s2_tiles"
!python scripts/prepare_liss4.py --scene /kaggle/working/s2/scene.tif --sensor sentinel2 --upsample 4 --min-width-m 8 --out $P --center "$CENTER" --size-km $SIZE_KM --cache /kaggle/working/osm

[liss4] bands: ['scene.tif']
[liss4] window 9600x9600 px at 2.50 m/px (native 10.00 m)
[liss4] OSM roads kept=33135 dropped=329542 road pixels=3.42%
[liss4] tiles: {'train': 198, 'val': 54, 'test': 72}
[liss4] wrote /kaggle/working/s2_tiles


In [8]:
# 1) Zero-shot: DeepGlobe-trained model applied directly
!python scripts/evaluate_checkpoint.py --checkpoint "$chosen" --data $P --out /kaggle/working/out/zero_shot --skip-occlusion --skip-classical --tile 512

  Post-training evaluation
  checkpoint : /kaggle/input/datasets/kashaksingh/rr-code/route-resilience-ai/checkpoints/segformer_b3_ft/best.pt
  arch       : segformer_b3   epoch 21   best cldice=0.6833
  device     : cuda   threshold 0.5   tile 512
  git        : None

[val] 54 tiles
       IoU 0.23713  Dice 0.33252  P 0.3561  R 0.35621  clDice 0.35313  PixAcc 0.96185

[test] 72 tiles
       IoU 0.2246  Dice 0.31035  P 0.33643  R 0.35005  clDice 0.32481  PixAcc 0.97161

  val - test IoU gap: +0.01253  (consistent)

Wrote /kaggle/working/out/zero_shot/evaluation_report.json and .md

# Evaluation report

- checkpoint: `/kaggle/input/datasets/kashaksingh/rr-code/route-resilience-ai/checkpoints/segformer_b3_ft/best.pt`
- architecture: `segformer_b3`  |  epoch 21  |  selected on **cldice** = 0.6833
- threshold 0.5  |  tile 512  |  seed 1234

## 1-2. Validation and held-out test

| split | n | iou | dice | precision | recall | cldice | pixel_accuracy |
|---|---|---|---|---|---|---|---|
| val 

In [9]:
# 2) Fine-tune on Hyderabad tiles with OSM labels (starts from the final SegFormer)
FT = "/kaggle/working/out/segformer_s2"
!python scripts/kaggle_train.py --data $P --arch segformer_b3 --init-from "$chosen" --out $FT --epochs 30 --tile-size 512 --lr 6e-5 --pos-weight 3 --lovasz-weight 0.5

  Route Resilience - segformer_b3 / DeepGlobe training

[1/7] Dependencies
  OK    segmentation-models-pytorch importable   [0.6.0]
  OK    torch available   [2.11.0+cu130]
  OK    CUDA device present   [Tesla T4]

[2/7] Dataset structure
  OK    data root exists   [/kaggle/working/s2_tiles]
  OK    train/images and train/masks exist
  OK    val/images and val/masks exist
  OK    test/images and test/masks exist

[3/7] Counts
        train  images=198    masks=198    pairs=198
        val    images=54     masks=54     pairs=54
        test   images=72     masks=72     pairs=72
  OK    train split is non-empty   [198 pairs]
  OK    val split is non-empty   [54 pairs]

[4/7] Pairing
  OK    every image has a mask   [0 unpaired: []]
  OK    image and mask dimensions agree   [0 mismatched: []]

[5/7] Mask values
  OK    masks are binary   [0 non-binary: []]
  OK    road pixel fraction is plausible   [0.0351 (DeepGlobe is typically 0.02-0.08)]
        mean road fraction 0.0351 -> background

In [10]:
# 3) Evaluate on the held-out (east) test strip
!python scripts/evaluate_checkpoint.py --checkpoint $FT/best.pt --data $P --out $FT/report --skip-occlusion --skip-classical --tile 512

  Post-training evaluation
  checkpoint : /kaggle/working/out/segformer_s2/best.pt
  arch       : segformer_b3   epoch 24   best cldice=0.67829
  device     : cuda   threshold 0.5   tile 512
  git        : None

[val] 54 tiles
       IoU 0.47209  Dice 0.60957  P 0.56271  R 0.68424  clDice 0.67829  PixAcc 0.97693

[test] 72 tiles
       IoU 0.46023  Dice 0.5927  P 0.54085  R 0.70777  clDice 0.64866  PixAcc 0.98075

  val - test IoU gap: +0.01186  (consistent)

Wrote /kaggle/working/out/segformer_s2/report/evaluation_report.json and .md

# Evaluation report

- checkpoint: `/kaggle/working/out/segformer_s2/best.pt`
- architecture: `segformer_b3`  |  epoch 24  |  selected on **cldice** = 0.67829
- threshold 0.5  |  tile 512  |  seed 1234

## 1-2. Validation and held-out test

| split | n | iou | dice | precision | recall | cldice | pixel_accuracy |
|---|---|---|---|---|---|---|---|
| val | 54 | 0.4721 | 0.6096 | 0.5627 | 0.6842 | 0.6783 | 0.9769 |
| test | 72 | 0.4602 | 0.5927 | 0.5409 | 0

In [11]:
shutil.copy(P + "/manifest.json", "/kaggle/working/out/s2_manifest.json")
shutil.copy(P + "/preview_osm_overlay.jpg", "/kaggle/working/out/s2_preview_osm_overlay.jpg")
shutil.copy("/kaggle/working/s2/scene.json", "/kaggle/working/out/s2_scene.json")
if os.path.exists(FT + "/final.pt"): os.remove(FT + "/final.pt")
shutil.make_archive("/kaggle/working/s2_results", "zip", "/kaggle/working/out")
shutil.rmtree(P, ignore_errors=True)
print("DONE")

DONE
